#FUNCTION

In [ ]:

%pip install beautifulsoup4
%pip install selenium
%pip install selenium_stealth

In [ ]:
import html
import json
import os
import re
import time, tempfile
from bs4 import BeautifulSoup
import bs4
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium_stealth import stealth

def scroll_to_bottom(driver, scroll_attempts=10, scroll_step=1000, delay=0.2):
    current_height = 0
    for i in range(scroll_attempts):
        driver.execute_script(f"window.scrollBy(0, {scroll_step});")
        time.sleep(delay)
        new_height = driver.execute_script("return document.documentElement.scrollTop")
        if new_height == current_height:
            break
        current_height = new_height

def setup_webdriver():
    chrome_options = Options()
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--headless")
    user_data_dir = tempfile.mkdtemp()
    chrome_options.add_argument(f"--user-data-dir={user_data_dir}")

    driver = webdriver.Chrome(options=chrome_options)

    stealth(driver,
        languages=["vi-VN", "vi"],
        vendor="Google Inc.",
        platform="Win32",
        webgl_vendor="Intel Inc.",
        renderer="Intel Iris OpenGL Engine",
        fix_hairline=True,
    )

    return driver

def generate_seo_fields(name: str):
    meta_title = name.strip()
    meta_description = f"Mua {name.strip()} giá tốt, chính hãng, bảo hành đầy đủ tại cửa hàng của chúng tôi."
    common_words = {"chính", "hãng", "vn/a", "|"}
    words = [w.lower() for w in re.split(r"\W+", name) if w.lower() not in common_words and len(w) > 1]
    meta_keywords = ", ".join(dict.fromkeys(words))
    slug = name.lower()
    slug = re.sub(r"[^\w\s-]", "", slug)
    slug = re.sub(r"\s+", "-", slug)
    slug = slug.strip("-")
    return {
        "UrlSlug": slug,
        "MetaTitle": meta_title,
        "MetaDescription": meta_description,
        "MetaKeywords": meta_keywords
    }

def crawl_menu_links(url):
    driver = setup_webdriver()
    menu_link = {}
    try:
        driver.get(url)
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".label-menu-tree"))
        )
        soup = bs4.BeautifulSoup(driver.page_source, "html.parser")
        menu_tree_elems = soup.select('.label-menu-tree')
        print(len(menu_tree_elems))
        for menu_elem in menu_tree_elems:
            if menu_elem.select("div.label-item.multiple"):
                link_elems = menu_elem.select("a.multiple-link")
                for link_elem in link_elems:
                    link = link_elem.get("href")
                    cat = link_elem.find("span").text.strip()
                    if cat not in menu_link:
                        menu_link[cat] = link
            else:
                link_elem = menu_elem.select_one("a.label-item")
                link = link_elem.get("href")
                cat = link_elem.find("span").text.strip()
                if cat not in menu_link:
                    menu_link[cat] = link
        print(len(menu_link))
    except Exception as e:
        print(f"Error during crawling: {e}")
    finally:
        driver.quit()
    return menu_link

def select_descriptions(soup):
    box_ksp = soup.select("div.box-ksp")
    main_img_url = ""
    li_items_text = []
    video_url = ""

    for box in box_ksp:
        try:
            # Lấy video nếu chưa có
            if not video_url:
                iframe = box.select_one("iframe")
                if iframe and 'src' in iframe.attrs:
                    video_url = iframe['src']

            # Lấy ảnh chính nếu chưa có
            if not main_img_url:
                main_img = box.select_one("img")
                if main_img:
                    main_img_url = main_img.get('src') or main_img.get('data-src', '')

            # Lấy descriptions nếu chưa có
            if not li_items_text:
                div_desktop = box.select_one("div.desktop")
                if div_desktop:
                    li_items_text = [li.text.strip() for li in div_desktop.select("li")]

        except Exception as e:
            print(f"{e}, tiep tuc")

    return li_items_text, main_img_url, video_url

def select_imgs(soup, max_images=5):
    thumbnail_slide = soup.select_one("div.thumbnail-slide div.swiper-wrapper")
    image_urls = []
    if not thumbnail_slide:
        print("Không tìm thấy thumbnail slide.")
        return image_urls
    img_tags = thumbnail_slide.select("img")
    for img in img_tags:
        if img:
            if 'src' in img.attrs:
                image_urls.append(img['src'])
            elif 'data-src' in img.attrs:
                image_urls.append(img['data-src'])
        else:
            image_urls.append("")
            print("Không tìm thấy ảnh")
        if len(image_urls) >= max_images:
            break
    return image_urls

def crawl_product_attribute(url):
    driver = setup_webdriver()
    attributes = []
    descriptions = []
    main_img_url = ""
    imgs = []
    video_url = ""

    try:
        print("Đang mở trang web... {}".format(url))
        driver.get(url)
        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")

        descriptions, main_img_url, video_url = select_descriptions(soup)
        imgs = select_imgs(soup)

        print("Đang cuộn trang...")
        scroll_to_bottom(driver)

        selector = ".button__show-modal-technical"
        try:
            print("Đang tìm nút 'Xem cấu hình chi tiết'...")
            load_more_button = driver.find_element(By.CSS_SELECTOR, selector)
            driver.execute_script("arguments[0].scrollIntoView(true);", load_more_button)
            time.sleep(3)
            driver.execute_script("arguments[0].click();", load_more_button)
            print("Đã click nút 'Xem cấu hình chi tiết'")
        except Exception as e:
            print("Không tìm thấy hoặc không click được nút:", e)
            driver.save_screenshot("error_click_button.png")
            return attributes, descriptions, main_img_url, imgs

        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        

        # Cấu trúc mới: section.technical-content-section > table.technical-content > tr.technical-content-item
        sections = soup.select("section.technical-content-section")
        if not sections:
            print("Không tìm thấy modal thông tin kỹ thuật.")
            return attributes, descriptions, main_img_url, imgs

        i = 0
        for section in sections:
            title_elem = section.select_one("p.title")
            type_name = title_elem.text.strip() if title_elem else "Khác"

            rows = section.select("tr.technical-content-item")
            for row in rows:
                try:
                    tds = row.select("td")
                    if len(tds) < 2:
                        continue
                    name = tds[0].get_text(strip=True)
                    # value có thể chứa <br> hoặc nhiều dòng
                    value_html = tds[1].decode_contents()
                    value = value_html.replace("<br/>", "\n").replace("<br>", "\n")
                    value = BeautifulSoup(value, "html.parser").get_text(separator="\n").strip()
                    attributes.append({
                        "name": name,
                        "value": value,
                        "displayorder": i,
                        "type": type_name
                    })
                    i += 1
                except Exception as e:
                    print(f"Lỗi khi xử lý thuộc tính: {e}")

    except Exception as e:
        print(f"Lỗi khi tải trang hoặc xử lý nút: {e}")
        driver.save_screenshot("error_general.png")
    finally:
        driver.quit()

    return attributes, descriptions, main_img_url, imgs, video_url

def extract_products(soup, cat):
    products = []
    product_items = soup.select('div.product-item')

    if not product_items:
        print("Không tìm thấy sản phẩm. Kiểm tra lại selector.")
        return products

    for item in product_items:
        try:
            name_elem = item.select_one('.product__name')
            name = name_elem.text.strip() if name_elem else "Unknown"

            url_elem = item.select_one('a.product__link')
            url = url_elem['href'] if url_elem and 'href' in url_elem.attrs else ""

            price_elem = item.select_one('.product__price--show')
            price = price_elem.text.strip() if price_elem else "0"
            price = price.replace('đ', '').replace('.', '').strip()

            old_price_elem = item.select_one('.product__price--through')
            old_price = old_price_elem.text.strip() if old_price_elem else "0"
            old_price = old_price.replace('đ', '').replace('.', '').strip() if old_price else "0"

            discount_elem = item.select_one('.product__price--percent-detail')
            text = html.unescape(discount_elem.text) if discount_elem else "0"
            match = re.search(r'\d+', text)
            discount = match.group() if match else "0"

            tags = []
            tags_elem = item.select('.product__more-info__item')
            for tag_elem in tags_elem:
                if "hàng" not in tag_elem.text:
                    tags.append(tag_elem.text)

            img_elem = item.select_one('.product__img')
            img_url = ""
            if img_elem:
                if 'src' in img_elem.attrs and img_elem['src'] and 'data:image' not in img_elem['src']:
                    img_url = img_elem['src']
                elif 'data-src' in img_elem.attrs:
                    img_url = img_elem['data-src']

            brand = "Unknown"
            common_brands = ["Samsung", "Apple", "iPad", "Xiaomi", "Lenovo", "Huawei", "Nokia", "Teclast", "TCL"]
            for b in common_brands:
                if b.lower() in name.lower():
                    brand = b
                    if b == "iPad":
                        brand = "Apple"
                    break

            seo_fields = generate_seo_fields(name)
            price_int = int(price) if price.isdigit() else 0
            old_price_int = int(old_price) if (old_price.isdigit() and int(old_price) > 0) else price_int
            product = {
                'name': name,
                'url': url,
                'price': price_int,
                'old_price': old_price_int,
                'discount': int(discount) if discount.isdigit() else 0,
                'image_url': img_url,
                'brand': brand,
                'category': cat,
                'tags': tags,
                'quantityinstock': 200,
                'urlslug': seo_fields['UrlSlug'],
                'metatitle': seo_fields['MetaTitle'],
                'metadescription': seo_fields['MetaDescription'],
                'metakeywords': seo_fields['MetaKeywords'],
            }
            if product['price'] > 0:
                products.append(product)

        except Exception as e:
            print(f"Error extracting product: {e}")
    return products

def crawl_with_selenium(url, cat, fileName, max_load_more=5):
    print(f"Starting Selenium to crawl: {url}")
    driver = setup_webdriver()
    all_products = []

    try:
        driver.get(url)
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.product-item"))
        )

        load_more_count = 0
        while load_more_count < max_load_more:
            print("Analyzing current page...")
            soup = bs4.BeautifulSoup(driver.page_source, "html.parser")
            new_products = extract_products(soup, cat)
            print(f"Found {len(new_products)} products. Total so far: {len(all_products) + len(new_products)}")

            for product in new_products:
                if product not in all_products:
                    all_products.append(product)

            try:
                print("Looking for 'Xem thêm' button...")
                selectors = [
                    "a.button__show-more-product",
                    ".cps-block-content_btn-showmore a",
                    ".btn-show-more"
                ]
                load_more_button = None
                for selector in selectors:
                    try:
                        load_more_button = WebDriverWait(driver, 3).until(
                            EC.element_to_be_clickable((By.CSS_SELECTOR, selector))
                        )
                        print(f"Found button with selector: {selector}")
                        break
                    except:
                        continue

                if not load_more_button:
                    print("Button not found with any selector")
                    break

                print(f"Button text: {load_more_button.text}")
                driver.execute_script("arguments[0].scrollIntoView(true);", load_more_button)
                time.sleep(2)
                driver.execute_script("arguments[0].click();", load_more_button)
                print("Clicked button using JavaScript")
                time.sleep(5)
                load_more_count += 1
                print(f"Clicked 'Load More' button ({load_more_count}/{max_load_more})")

            except Exception as e:
                print(f"Không thể click nút 'Xem thêm': {e}")
                break

    except Exception as e:
        print(f"Error during crawling: {e}")
    finally:
        driver.quit()

    unique_products = []
    unique_urls = set()
    for product in all_products:
        if product['url'] not in unique_urls:
            unique_urls.add(product['url'])
            unique_products.append(product)

    filename = fileName or '{}_data_full.json'.format(cat)
    os.makedirs(os.path.dirname(os.path.abspath(filename)), exist_ok=True)
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(unique_products, f, ensure_ascii=False, indent=4)

    print(f"Done! Crawled {len(unique_products)} unique products in total")
    return unique_products


#PHONE

##MAIN INFORMATIONS 

In [ ]:
#craw phone: main information first
crawl_with_selenium("https://cellphones.com.vn/mobile.html", "phone", "Json/phone_data_full.json")


##ATTRIBUTES

In [1]:
%pip install tqdm


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
from tqdm.notebook import tqdm

with open('D:\TechStore\server\ProductService\DataProcessing\Json\phone_data_full.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(type(data))  # <class 'list'>
attribute_json = []
for _ in data:
    # print(crawl_product_attribute(_['url']))
    attributes, descriptions, main_img_url, imgs = crawl_product_attribute(_['url'])
    image_list = []
    if main_img_url: 
        image_list.append(main_img_url)
    if imgs:  
        for img in imgs:
            image_list.append(img)
    if attributes:
        attribute_json.append({
            "url": _['url'],
            "attributes": attributes,
            "descriptions": descriptions,
            "imgs": image_list,
            "video_url": main
        })
    else:
        print(f"Không tìm thấy thuộc tính cho sản phẩm: {_['name']}")
print(attribute_json)
with open('D:\TechStore\server\ProductService\DataProcessing\Json\phone_attributes_data_full.json', 'w', encoding='utf-8') as f:
    json.dump(attribute_json, f, ensure_ascii=False, indent=4)

print("Đã lưu toàn bộ attributes vào attributes_data_full.json")

<>:4: SyntaxWarning: "\T" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\T"? A raw string is also an option.
<>:28: SyntaxWarning: "\T" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\T"? A raw string is also an option.
<>:4: SyntaxWarning: "\T" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\T"? A raw string is also an option.
<>:28: SyntaxWarning: "\T" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\T"? A raw string is also an option.
C:\Users\thaih\AppData\Local\Temp\ipykernel_3844\2959393598.py:4: SyntaxWarning: "\T" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\T"? A raw string is also an option.
  with open('D:\TechStore\server\ProductService\DataProcessing\Json\phone_data_full.json', 'r', encoding='utf-8') as f:
C:\Users\thaih\AppData\Local\Temp\ipykernel_3844\2959

<class 'list'>
Đang mở trang web... https://cellphones.com.vn/iphone-17-pro.html
Đang cuộn trang...
Đang tìm nút 'Xem cấu hình chi tiết'...
Đã click nút 'Xem cấu hình chi tiết'
Đang mở trang web... https://cellphones.com.vn/dien-thoai-samsung-galaxy-s26-ultra.html
Đang cuộn trang...
Đang tìm nút 'Xem cấu hình chi tiết'...
Đã click nút 'Xem cấu hình chi tiết'
Đang mở trang web... https://cellphones.com.vn/iphone-17-pro-max.html
Đang cuộn trang...
Đang tìm nút 'Xem cấu hình chi tiết'...
Đã click nút 'Xem cấu hình chi tiết'
Đang mở trang web... https://cellphones.com.vn/iphone-17-256gb.html
Đang cuộn trang...
Đang tìm nút 'Xem cấu hình chi tiết'...
Đã click nút 'Xem cấu hình chi tiết'
Đang mở trang web... https://cellphones.com.vn/dien-thoai-samsung-galaxy-s26.html
Đang cuộn trang...
Đang tìm nút 'Xem cấu hình chi tiết'...
Đã click nút 'Xem cấu hình chi tiết'
Đang mở trang web... https://cellphones.com.vn/dien-thoai-samsung-galaxy-a17-5g.html
Đang cuộn trang...
Đang tìm nút 'Xem cấu hình 

: 

#TABLET

In [7]:
#craw tablet: main information first
crawl_with_selenium("https://cellphones.com.vn/tablet.html", "tablet", "Json/tablet_data_full.json")


Starting Selenium to crawl: https://cellphones.com.vn/tablet.html
Analyzing current page...
Found 20 products. Total so far: 20
Looking for 'Xem thêm' button...
Found button with selector: a.button__show-more-product
Button text: Xem thêm 180 sản phẩm
Clicked button using JavaScript
Clicked 'Load More' button (1/5)
Analyzing current page...
Found 40 products. Total so far: 60
Looking for 'Xem thêm' button...
Found button with selector: a.button__show-more-product
Button text: Xem thêm 160 sản phẩm
Clicked button using JavaScript
Clicked 'Load More' button (2/5)
Analyzing current page...
Found 60 products. Total so far: 100
Looking for 'Xem thêm' button...
Found button with selector: a.button__show-more-product
Button text: Xem thêm 140 sản phẩm
Clicked button using JavaScript
Clicked 'Load More' button (3/5)
Analyzing current page...
Found 80 products. Total so far: 140
Looking for 'Xem thêm' button...
Found button with selector: a.button__show-more-product
Button text: Xem thêm 120 sả

[{'name': 'iPad A16 Wifi 128GB 2025 | Chính hãng Apple Việt Nam',
  'url': 'https://cellphones.com.vn/ipad-a16-11-inch.html',
  'price': 9290000,
  'old_price': 9990000,
  'discount': 7,
  'image_url': 'https://cdn2.cellphones.com.vn/insecure/rs:fill:358:358/q:90/plain/https://cellphones.com.vn/media/catalog/product/i/p/ipad-a16-11-inch_10_.jpg',
  'brand': 'Apple',
  'category': 'tablet',
  'tags': ['11 inches', '128 GB'],
  'quantityinstock': 200,
  'urlslug': 'ipad-a16-wifi-128gb-2025-chính-hãng-apple-việt-nam',
  'metatitle': 'iPad A16 Wifi 128GB 2025 | Chính hãng Apple Việt Nam',
  'metadescription': 'Mua iPad A16 Wifi 128GB 2025 | Chính hãng Apple Việt Nam giá tốt, chính hãng, bảo hành đầy đủ tại cửa hàng của chúng tôi.',
  'metakeywords': 'ipad, a16, wifi, 128gb, 2025, apple, việt, nam'},
 {'name': 'Honor Pad X8b Wifi 6GB 128GB',
  'url': 'https://cellphones.com.vn/may-tinh-bang-honor-pad-x8b.html',
  'price': 7290000,
  'old_price': 7490000,
  'discount': 3,
  'image_url': 'htt

In [8]:
#craw laptop: main information first
crawl_with_selenium("https://cellphones.com.vn/laptop.html", "laptop", "Json/laptop_data_full.json")


Starting Selenium to crawl: https://cellphones.com.vn/laptop.html
Analyzing current page...
Found 20 products. Total so far: 20
Looking for 'Xem thêm' button...
Found button with selector: a.button__show-more-product
Button text: Xem thêm 688 sản phẩm
Clicked button using JavaScript
Clicked 'Load More' button (1/5)
Analyzing current page...
Found 40 products. Total so far: 60
Looking for 'Xem thêm' button...
Found button with selector: a.button__show-more-product
Button text: Xem thêm 668 sản phẩm
Clicked button using JavaScript
Clicked 'Load More' button (2/5)
Analyzing current page...
Found 60 products. Total so far: 100
Looking for 'Xem thêm' button...
Found button with selector: a.button__show-more-product
Button text: Xem thêm 648 sản phẩm
Clicked button using JavaScript
Clicked 'Load More' button (3/5)
Analyzing current page...
Found 80 products. Total so far: 140
Looking for 'Xem thêm' button...
Found button with selector: a.button__show-more-product
Button text: Xem thêm 628 sả

[{'name': 'Laptop MSI Prestige 13 AI+ Ukiyoe Edition A2VMG-075VN',
  'url': 'https://cellphones.com.vn/laptop-msi-prestige-13-ai-ukiyoe-edition-a2vmg-075vn.html',
  'price': 51990000,
  'old_price': 51990000,
  'discount': 0,
  'image_url': 'https://cdn2.cellphones.com.vn/insecure/rs:fill:358:358/q:90/plain/https://cellphones.com.vn/media/catalog/product/g/r/group_881_1.png',
  'brand': 'Unknown',
  'category': 'laptop',
  'tags': [],
  'quantityinstock': 200,
  'urlslug': 'laptop-msi-prestige-13-ai-ukiyoe-edition-a2vmg-075vn',
  'metatitle': 'Laptop MSI Prestige 13 AI+ Ukiyoe Edition A2VMG-075VN',
  'metadescription': 'Mua Laptop MSI Prestige 13 AI+ Ukiyoe Edition A2VMG-075VN giá tốt, chính hãng, bảo hành đầy đủ tại cửa hàng của chúng tôi.',
  'metakeywords': 'laptop, msi, prestige, 13, ai, ukiyoe, edition, a2vmg, 075vn'},
 {'name': 'Laptop HP Omnibook 5 AI 16-AF1048TU BZ7Q9PA',
  'url': 'https://cellphones.com.vn/laptop-hp-omnibook-5-ai-16-af1048tu-bz7q9pa.html',
  'price': 25990000,